## What this project is

**marisco** is a data curation tool developed at the IAEA Marine Environmental Laboratories (Monaco) for [MARIS](https://maris.iaea.org), the IAEA's open-access marine radioactivity repository.

**The problem it solves:** Marine radioactivity data comes from many providers: regional monitoring programmes (HELCOM, OSPAR), event-driven datasets (TEPCO data following Fukushima), individual research papers, and more. Each uses different file formats, nomenclature, units, and detection-limit conventions. MARIS ingests all of them into a single central database, but aligning them requires significant curation work.

**What marisco does:** It replaces a manual OpenRefine-based curation workflow with a reproducible Python pipeline. For each dataset, marisco:

1. Reads the raw provider data in whatever format it arrives
2. Aligns it to the MARIS data schema, standardising nomenclature, units, detection levels, and sample-type classification
3. Encodes the curated dataset as a self-contained **NetCDF4 file** that bundles measurements, variable metadata, lookup tables of used nomenclatures, and bibliographic global attributes in a single file
4. Can also export `.csv` files compatible with the existing OpenRefine → MARIS central-DB import pipeline

**How MARIS data is disseminated:** Via the web interface (https://maris.iaea.org), a data API, and as NetCDF files — marisco-generated NetCDF files feed all three channels.

## Writing style

**No em-dashes.** Use periods or commas instead. Write in plain expository prose. Use simple sentences.

For instance:

- BAD: The HELCOM handler loads raw data — then applies unit conversions — before aligning to the MARIS schema.
- GOOD: The HELCOM handler loads raw data, applies unit conversions, and then aligns to the MARIS schema.

- BAD: The row has two value columns — VALUE_Bq/kg and VALUE_Bq/m² — so it must be split.
- GOOD: The row has two value columns (VALUE_Bq/kg and VALUE_Bq/m²), so it must be split.

- BAD: This means every sediment row with data in both columns must be split into two rows during ingestion — an extra transformation step that a long-format delivery would avoid.
- GOOD: This means every sediment row with data in both columns must be split into two rows during ingestion, an extra transformation step that a long-format delivery would avoid.

## Critical rule for nbdev

**Never edit `.py` files in `marisco/`. They are auto-generated from notebooks.** All code lives in `nbs/`. After editing a notebook, run `nbdev-export` to regenerate modules.

Documentation follows the [`fastcore.docments`](https://fastcore.fast.ai/docments.html) convention: parameter documentation lives inline with the argument, not in a docstring body. nbdev picks these up automatically and renders them into the Quarto-based documentation site.

```python
def draw_n(n:int,        # Number of cards to draw
           replace:bool=True  # Draw with replacement?
          )->list:        # List of cards
    "Draw `n` cards."
```

Always use this style — inline `#` comments after type annotations — rather than numpy/Google-style docstrings.

## The four sample type groups

All measurements belong to one of four groups (also the NetCDF4 groups within each output file):

- `SEAWATER` — dissolved/filtered water samples (Bq/m³)
- `BIOTA` — marine organisms (Bq/kg wet or dry weight)
- `SEDIMENT` — bottom sediments (Bq/kg or Bq/m²)
- `SUSPENDED_MATTER` — suspended particles

## Key notebook locations

```
nbs/index.ipynb                      ← package overview and getting started
nbs/api/callbacks.ipynb              ← Callback, Transformer, built-in callbacks
nbs/api/configs.ipynb                ← NC_VARS, NC_GROUPS, NC_DTYPES, LUT helpers
nbs/api/metadata.ipynb               ← GlobAttrsFeeder, ZoteroCB, BboxCB
nbs/api/encoders.ipynb               ← NetCDFEncoder
nbs/api/decoders.ipynb               ← NetCDFDecoder
nbs/api/utils.ipynb                  ← ExtractNetcdfContents, general utilities
nbs/api/geo.ipynb                    ← ddmm_to_dd, other geo utilities
nbs/api/match.ipynb                  ← Matching / lookup-table helpers
nbs/api/netcdf2csv.ipynb             ← NetCDF-to-CSV export
nbs/api/files/cdl/maris.cdl         ← NetCDF4 template definition (CDL)
nbs/api/files/nc/maris-template.nc  ← NetCDF4 template (committed binary)
nbs/handlers/                       ← Contains all handlers: helcom, tepco, orspar, geotraces, maris_legacy, ...
nbs/cli/to_nc.ipynb                  ← maris_to_nc
nbs/cli/db_to_nc.ipynb               ← maris_db_to_nc
nbs/cli/nc_to_csv.ipynb              ← maris_nc_to_csv
```


## How the package works

Each data provider has a **handler** (`nbs/handlers/*.ipynb`). Every handler exposes an `encode(fname_out)` function that:

1. Loads raw provider data → `Dict[str, pd.DataFrame]` (one per sample type group)
2. Runs a `Transformer` with an ordered list of `Callback` objects that standardise the data
3. Feeds transformed data to `GlobAttrsFeeder` for NetCDF global attributes (bbox, time range, Zotero bibliographic metadata)
4. Writes output via `NetCDFEncoder`

## CLI tools

The CLI commands (`maris_init`, `maris_to_nc`, etc.) are defined in `nbs/cli/` and built with [`fastcore.script`](https://fastcore.fast.ai/script.html). The `@call_parse` decorator on a function generates the CLI entry point — arguments are inferred from the function signature. Entry points are declared in `pyproject.toml` under `[project.scripts]`.

## Go deeper

- when a user asks about writing or refactoring individual functions and needs guidance on naming conventions, abbreviations, and fastai/fastcore idioms used in this codebase — load [CRAFTs/coding-style-abbr.ipynb](CRAFTs/coding-style-abbr.ipynb)
- when a user asks about documenting a handler or wants to see the handler doc template — load [CRAFTs/handler-doc-style.ipynb](CRAFTs/handler-doc-style.ipynb)
- when a user asks about high-level software architecture, design principles, abstraction layers, or system structure (not just code-level style) — load [CRAFTs/sicp-design-memento.ipynb](CRAFTs/sicp-design-memento.ipynb)
